In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
new_df = pd.read_csv("train.csv")

In [ ]:
new_df.dropna(inplace=True)

In [ ]:
new_df.drop_duplicates(inplace=True)

In [ ]:
import re
import string

CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "'t": " not",
    "'ve": " have",
    "'m": " am",
}


def preprocess_text(text, stem=False):
    """Clean raw text and prepare it for feature extraction.

    Steps:
    - lowercasing
    - normalize contractions
    - remove HTML tags, URLs, emails, mentions, and hashtags
    - remove punctuation, digits, and extra whitespace
    - optional stemming when nltk is available
    """
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    for old, new in CONTRACTIONS.items():
        text = text.replace(old, new)

    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[@#]\w+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    if stem:
        try:
            from nltk.stem import PorterStemmer
            stemmer = PorterStemmer()
            tokens = [stemmer.stem(token) for token in tokens]
        except Exception:
            pass

    return " ".join(tokens)


sample_text = "<p>How can I improve my coding skills? Visit https://example.com now! I can't wait :)</p>"
preprocess_text(sample_text)

In [ ]:
new_df['question1'] = new_df['question1'].apply(preprocess_text)
new_df['question2'] = new_df['question2'].apply(preprocess_text)

In [ ]:
new_df.head()

In [ ]:
new_df.drop(columns=['id', 'qid1', 'qid2'], axis= 1)

In [ ]:
all_questions = list(new_df['question1']) + list(new_df['question2'])

In [ ]:
#Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tok = Tokenizer(num_words=200000)
tok.fit_on_texts(all_questions)
seq1 = tok.texts_to_sequences(new_df['question1'])
seq2 = tok.texts_to_sequences(new_df['question2'])

In [ ]:
padded1 = pad_sequences(seq1, maxlen=25, padding='post')
padded2 = pad_sequences(seq2, maxlen=25, padding='post')

In [ ]:
from sklearn.model_selection import train_test_split

y = new_df['is_duplicate'].astype(int).values

X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    padded1,
    padded2,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X1_train:', X1_train.shape)
print('X1_test :', X1_test.shape)
print('X2_train:', X2_train.shape)
print('X2_test :', X2_test.shape)
print('y_train :', y_train.shape)
print('y_test  :', y_test.shape)

Finding good maxlen

In [ ]:
lengths = [len(x) for x in seq1 + seq2]

print(np.percentile(lengths, 90))
print(np.percentile(lengths, 95))
print(np.max(lengths))

BiLSTM Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Multiply, Concatenate, Lambda, GlobalMaxPool1D
from tensorflow.keras.optimizers import Adam

MAXLEN = 25
VOCAB_SIZE = len(tok.word_index) + 1

input1 = Input(shape=(MAXLEN,))
input2 = Input(shape=(MAXLEN,))

embedding_layer = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=64
)

shared_lstm = Bidirectional(
    LSTM(
        64,
        return_sequences=True,
        dropout=0.2,
        recurrent_dropout=0.0   # important: remove recurrent_dropout for stability
    )
)

encoded1 = embedding_layer(input1)
encoded1 = shared_lstm(encoded1)
encoded1 = GlobalMaxPool1D()(encoded1)

encoded2 = embedding_layer(input2)
encoded2 = shared_lstm(encoded2)
encoded2 = GlobalMaxPool1D()(encoded2)

l1_distance = Lambda(lambda tensors: tf.abs(tensors[0] - tensors[1]))([encoded1, encoded2])
merged = Concatenate()([encoded1, encoded2, l1_distance])

x = Dense(128, activation='relu')(merged)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=[input1, input2], outputs=output)

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    [X1_train, X2_train],
    y_train,
    # better than validation_split here
    validation_data=([X1_test, X2_test], y_test),
    epochs=3,
    batch_size=128   # much safer than 512
)

In [ ]:
loss, accuracy = model.evaluate(
    [X1_test, X2_test],
    y_test
)

print('Accuracy:', accuracy)

In [ ]:
def predict_duplicate(q1, q2):

    q1 = preprocess_text(q1)
    q2 = preprocess_text(q2)

    q1_seq = tok.texts_to_sequences([q1])
    q2_seq = tok.texts_to_sequences([q2])

    q1_pad = pad_sequences(
        q1_seq,
        maxlen=MAXLEN,
        padding='post'
    )

    q2_pad = pad_sequences(
        q2_seq,
        maxlen=MAXLEN,
        padding='post'
    )

    pred = model.predict([q1_pad, q2_pad])[0][0]

    print('Duplicate Probability:', pred)

    if pred > 0.5:
        print('Duplicate Questions')
    else:
        print('Not Duplicate')

In [ ]:
predict_duplicate(
    'How can I learn machine learning?',
    'What is the best way to study ML?'
)

In [ ]:
import pickle

pickle.dump(model, open('BiLSTM_Model.pkl', 'wb'))

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
new_df = pd.read_csv("train.csv")

In [4]:
new_df.dropna(inplace=True)

In [5]:
new_df.drop_duplicates(inplace=True)

In [6]:
import re
import string

CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "'t": " not",
    "'ve": " have",
    "'m": " am",
}


def preprocess_text(text, stem=False):
    """Clean raw text and prepare it for feature extraction.

    Steps:
    - lowercasing
    - normalize contractions
    - remove HTML tags, URLs, emails, mentions, and hashtags
    - remove punctuation, digits, and extra whitespace
    - optional stemming when nltk is available
    """
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    for old, new in CONTRACTIONS.items():
        text = text.replace(old, new)

    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"[@#]\w+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    if stem:
        try:
            from nltk.stem import PorterStemmer
            stemmer = PorterStemmer()
            tokens = [stemmer.stem(token) for token in tokens]
        except Exception:
            pass

    return " ".join(tokens)


sample_text = "<p>How can I improve my coding skills? Visit https://example.com now! I can't wait :)</p>"
preprocess_text(sample_text)

'how can i improve my coding skills visit now i cannot wait'

In [7]:
new_df['question1'] = new_df['question1'].apply(preprocess_text)
new_df['question2'] = new_df['question2'].apply(preprocess_text)

In [8]:
new_df.head()

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,1,3,4,what is the story of kohinoor kohinoor diamond,what would happen if the indian government sto...,0
2,2,5,6,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,3,7,8,why am i mentally very lonely how can i solve it,find the remainder when math math is divided by,0
4,4,9,10,which one dissolve in water quikly sugar salt ...,which fish would survive in salt water,0


In [9]:
new_df.drop(columns=['id', 'qid1', 'qid2'], axis= 1)

,question1,question2,is_duplicate
0,what is the step by step guide to invest in sh...,what is the step by step guide to invest in sh...,0
1,what is the story of kohinoor kohinoor diamond,what would happen if the indian government sto...,0
2,how can i increase the speed of my internet co...,how can internet speed be increased by hacking...,0
3,why am i mentally very lonely how can i solve it,find the remainder when math math is divided by,0
4,which one dissolve in water quikly sugar salt ...,which fish would survive in salt water,0
...,...,...,...
404285,how many keywords are there in the racket prog...,how many keywords are there in perl programmin...,0
404286,do you believe there is life after death,is it true that there is life after death,1
404287,what is one coin,what is this coin,0
404288,what is the approx annual cost of living while...,i am having little hairfall problem but i want...,0


In [10]:
all_questions = list(new_df['question1']) + list(new_df['question2'])

In [11]:
#Tokenization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
tok = Tokenizer(num_words=200000)
tok.fit_on_texts(all_questions)
seq1 = tok.texts_to_sequences(new_df['question1'])
seq2 = tok.texts_to_sequences(new_df['question2'])

In [12]:
padded1 = pad_sequences(seq1, maxlen=25, padding='post')
padded2 = pad_sequences(seq2, maxlen=25, padding='post')

In [13]:
from sklearn.model_selection import train_test_split

y = new_df['is_duplicate'].astype(int).values

X1_train, X1_test, X2_train, X2_test, y_train, y_test = train_test_split(
    padded1,
    padded2,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X1_train:', X1_train.shape)
print('X1_test :', X1_test.shape)
print('X2_train:', X2_train.shape)
print('X2_test :', X2_test.shape)
print('y_train :', y_train.shape)
print('y_test  :', y_test.shape)

X1_train: (323429, 25)
X1_test : (80858, 25)
X2_train: (323429, 25)
X2_test : (80858, 25)
y_train : (323429,)
y_test  : (80858,)


Finding good maxlen

In [14]:
lengths = [len(x) for x in seq1 + seq2]

print(np.percentile(lengths, 90))
print(np.percentile(lengths, 95))
print(np.max(lengths))

18.0
23.0
244


BiLSTM Model

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Multiply, Concatenate, Lambda, GlobalMaxPool1D
from tensorflow.keras.optimizers import Adam

MAXLEN = 25
VOCAB_SIZE = len(tok.word_index) + 1

input1 = Input(shape=(MAXLEN,))
input2 = Input(shape=(MAXLEN,))

embedding_layer = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=64
)

shared_lstm = Bidirectional(
    LSTM(
        64,
        return_sequences=True,
        dropout=0.2,
        recurrent_dropout=0.0   # important: remove recurrent_dropout for stability
    )
)

encoded1 = embedding_layer(input1)
encoded1 = shared_lstm(encoded1)
encoded1 = GlobalMaxPool1D()(encoded1)

encoded2 = embedding_layer(input2)
encoded2 = shared_lstm(encoded2)
encoded2 = GlobalMaxPool1D()(encoded2)

l1_distance = Lambda(lambda tensors: tf.abs(
    tensors[0] - tensors[1]))([encoded1, encoded2])
merged = Concatenate()([encoded1, encoded2, l1_distance])

x = Dense(128, activation='relu')(merged)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=[input1, input2], outputs=output)

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    [X1_train, X2_train],
    y_train,
    # better than validation_split here
    validation_data=([X1_test, X2_test], y_test),
    epochs=3,
    batch_size=128   # much safer than 512
)

Epoch 1/3
2527/2527 [==============================] - 119s 41ms/step - loss: 0.4535 - accuracy: 0.7788 - val_loss: 0.4124 - val_accuracy: 0.7985
Epoch 2/3
2527/2527 [==============================] - 96s 38ms/step - loss: 0.3455 - accuracy: 0.8428 - val_loss: 0.3918 - val_accuracy: 0.8171
Epoch 3/3
2527/2527 [==============================] - 101s 40ms/step - loss: 0.2914 - accuracy: 0.8708 - val_loss: 0.3943 - val_accuracy: 0.8259


In [27]:
loss, accuracy = model.evaluate(
    [X1_test, X2_test],
    y_test
)

print('Accuracy:', accuracy)

2527/2527 [==============================] - 33s 13ms/step - loss: 0.3943 - accuracy: 0.8259
Accuracy: 0.825941801071167


In [21]:
def predict_duplicate(q1, q2):

    q1 = preprocess_text(q1)
    q2 = preprocess_text(q2)

    q1_seq = tok.texts_to_sequences([q1])
    q2_seq = tok.texts_to_sequences([q2])

    q1_pad = pad_sequences(
        q1_seq,
        maxlen=MAXLEN,
        padding='post'
    )

    q2_pad = pad_sequences(
        q2_seq,
        maxlen=MAXLEN,
        padding='post'
    )

    pred = model.predict([q1_pad, q2_pad])[0][0]

    print('Duplicate Probability:', pred)

    if pred > 0.5:
        print('Duplicate Questions')
    else:
        print('Not Duplicate')

In [26]:
predict_duplicate(
    'How can I learn machine learning?',
    'What is the best way to study ML?'
)

1/1 [==============================] - 0s 168ms/step
Duplicate Probability: 0.10849226
Not Duplicate


In [25]:
model.save("BiLSTM.keras")